In [ ]:
import os
import sys
sys.path.insert(0, r"c:\repos\DroneDetectionRF")
from scipy import signal
import torch
import numpy as np
import torch
from NoisyUAV.modelos.burst_cvcnn import BurstCVCNN
from funciones.detector_entropia import detectar_bursts, plot_muestra, print_diagnostico, plot_espectrograma_3d
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:

def cargar_dronerf_14mhz(ruta_csv, fs_origen=40e6, fs_destino=14e6):
    """
    Lee CSV crudo de DroneRF, reconstruye el tensor complejo (I+jQ), 
    diezma polifásicamente a 14 MHz y lo empaqueta como Tensor PyTorch.
    """
    print(f"Leyendo CSV original (puede tardar unos segundos): {os.path.basename(ruta_csv)}")
    with open(ruta_csv, 'r') as f:
        csv_text = f.read()
        
    arr_1d = np.fromstring(csv_text, sep=',')
    n_cplx = len(arr_1d) // 2
    iq_origen = arr_1d[0:n_cplx*2:2] + 1j * arr_1d[1:n_cplx*2:2]
    
    print(f"[*] Señal Original: {len(iq_origen)} muestras de radar a {fs_origen/1e6} MHz")
    
    up = int(fs_destino / 1e6)
    down = int(fs_origen / 1e6)
    iq_14mhz = signal.resample_poly(iq_origen, up, down)
    
    print(f"[*] Señal Diezmada: {len(iq_14mhz)} muestras a {fs_destino/1e6} MHz")
    
    iq_final = np.stack((np.real(iq_14mhz), np.imag(iq_14mhz)), axis=0).astype(np.float32)
    return torch.from_numpy(iq_final)

In [ ]:
# 1. CARGAR LA MUESTRA EXTERNA RESAMPLEADA A 14 MHz
RUTA = r"c:\TFM_data\DroneRF\DroneRF\Phantom drone\RF Data_11000_H\RF Data_11000_H\11000H_10.csv"
iq_tensor = cargar_dronerf_14mhz(RUTA)

# Prueba inferencia modelo alumno

In [ ]:
# IMPORTANTE: Cargamos el modelo TEACHER (Oráculo entrenado en SNR >= 0)
ruta_pesos = r"c:\repos\DroneDetectionRF\NoisyUAV\curriculum_alumn_v1\checkpoints\alumn_model_best.pt"
ckpt = torch.load(ruta_pesos, map_location=device, weights_only=False)
model = BurstCVCNN().to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# Rescatamos las estadísticas físicas (mean/std) originales con las que se entrenó el Teacher
phys_mean = torch.tensor(ckpt['phys_mean'], dtype=torch.float32).to(device)
phys_std  = torch.tensor(ckpt['phys_std'],  dtype=torch.float32).to(device)
print(f"✅ Nuevo modelo TEACHER cargado con éxito en {device.type.upper()}")
print(f"   Época Óptima del guardado: {ckpt.get('epoch', 'N/A')}")
print(f"   Validation F1: {ckpt.get('val_f1', 0.0):.4f}")

In [ ]:
FS = 14e6
NPERSEG = 2048
Z_THRESH = 1.5      
MIN_BURST_MS = 0.5
MERGE_GAP_MS = 2.0
MIN_Z_ABS = 4.0
BG_MULT = 4
MAX_BINS_FRAC = 1
SMOOTH_MS = 0.1
ADAPTIVE_WINDOW_MS = 10 

t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
    iq_tensor, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    min_z_abs=MIN_Z_ABS, bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    smooth_ms=SMOOTH_MS, adaptive_window_ms=ADAPTIVE_WINDOW_MS,
)
# 3. VEREDICTO DEL SISTEMA DE ENTROPÍA (CFAR)
print_diagnostico(
    t_ms, nf_v, ns, umbral_v, n_active, bursts,
    nperseg=NPERSEG, fs=FS, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    target="AR Drone (DroneRF)", snr="H-Band / Desconocida", index=0,
)
# 4. VISUALIZACIONES INFERIORES
fig_2d = plot_muestra(
    iq_tensor, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
    fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    adaptive_window_ms=ADAPTIVE_WINDOW_MS,
    titulo="Extranjero (DroneRF): BUI=10100 (AR Drone - High Band)"
)
fig_2d.show()
fig_3d = plot_espectrograma_3d(
    iq_tensor, fs=FS, nperseg=NPERSEG, 
    t_lim_ms=None, smooth_sigma=2.0, floor_pct=5,
    titulo="Espectro Externo Procesado a 14 MHz"
)
# fig_3d.show()

In [ ]:
print("==================================================")
print("  VEREDICTO TEACHER (¿Sabe cazar drones reales?)  ")
print("==================================================")
drones_encontrados = 0
ruidos_encontrados = 0
if len(bursts) == 0:
    print("  ❌ No se detectaron ráfagas. La IA asume que la sala está vacía (RUIDO).")
else:
    # Parámetros Globales estáticos para toda la sala
    global_nf     = float(np.median(nf_v))
    global_ns     = float(np.clip(ns, 0, 5))
    global_H_mean = float(np.mean(H_smooth))
    global_p75_act= float(np.percentile(n_active, 75))
    
    with torch.no_grad():
        for i, b in enumerate(bursts):
            # 1. RECORTAR LA ONDA EXACTA
            idx_inicio = int(b['t0'] * 1e-3 * FS)
            idx_fin = int(b['t1'] * 1e-3 * FS)
            if idx_inicio >= idx_fin:
                continue
            pulso = iq_tensor[:, idx_inicio:idx_fin]
            
            # Normalización RMS de este latido
            power = pulso.pow(2).mean().clamp(min=1e-12).sqrt()
            pulso_normalizado = pulso / power
            
            # --- FIX CRÍTICO DE PADDING ---
            # Rellenar con ceros hasta 131072 muestras (9.4 ms) para simular 
            # el batch padding del entrenamiento y no saturar el AdaptiveAvgPool1d
            TARGET_LEN = 131072
            C, L = pulso_normalizado.shape
            if L < TARGET_LEN:
                pad = torch.zeros(C, TARGET_LEN - L, device=pulso_normalizado.device)
                pulso_padded = torch.cat([pulso_normalizado, pad], dim=1)
            else:
                pulso_padded = pulso_normalizado[:, :TARGET_LEN]
                
            input_ia = pulso_padded.unsqueeze(0).to(device) 
            # ------------------------------
            
            # 2. CONSTRUIR EL PERFIL FÍSICO (8 Dimensiones)
            dur_ms      = np.clip(b['dur_ms'], 0, 75)
            z_peak      = np.clip(abs(b['z_peak']), 0, 30)
            drop_b      = np.clip(b['drop_b'], 0, 10)
            n_act_burst = np.clip(b['n_act'], 0, 2048)
            
            feat_array = np.array([dur_ms, z_peak, drop_b, n_act_burst, 
                                   global_nf, global_ns, global_H_mean, global_p75_act], dtype=np.float32)
            
            feat_t = torch.from_numpy(feat_array).to(device)
            feat_norm = torch.clamp((feat_t - phys_mean) / (phys_std + 1e-8), -5.0, 5.0).unsqueeze(0)
            
            # 3. JUZGADO HÍBRIDO
            logit = model(input_ia, feat_norm)
            prob_dron = torch.sigmoid(logit).item() * 100 
            
            t_ms_inicio = b['t0']
            if prob_dron > 50.0:
                drones_encontrados += 1
                print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | z_peak={b['z_peak']:5.1f} | 🧠 IA: DRON   [{prob_dron:6.2f}%] ✅")
            else:
                ruidos_encontrados += 1
                print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | z_peak={b['z_peak']:5.1f} | 🧠 IA: RUIDO  [{prob_dron:6.2f}%] ❌")


In [ ]:
# ==========================================================
# EL CONMUTADOR MÁGICO (TRUE = Normalizado / FALSE = Crudo)
# ==========================================================
USAR_MODELO_NORMALIZADO = True

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ComplexConv1DNet(num_classes=2, pool_output_size=64, dropout=0.5)
if USAR_MODELO_NORMALIZADO:
    ruta_pesos = r"C:\TFM_data\NoisyUAV\stage2_norm\checkpoints\cvcnn_best_model.pth"
    print("Cargando Modelo UNIVERSAL NORMALIZADO...")
else:
    ruta_pesos = r"C:\TFM_data\NoisyUAV\stage2\checkpoints\cvcnn_best_model.pth"
    print("Cargando Modelo BASE (Sesgado al hardware local)...")
model.load_state_dict(torch.load(ruta_pesos, map_location=device, weights_only=True))
model.to(device)
model.eval()

In [ ]:
# 1. RECORTAMOS LOS BURSTS DE MEMORIA
muestras_recortadas = []
for b in bursts:
    idx_inicio = int(b['t0'] * 1e-3 * FS)
    idx_fin = int(b['t1'] * 1e-3 * FS)
    
    pulso = iq_adaptado[:, idx_inicio:idx_fin]
    muestras_recortadas.append(pulso)
print(f"✅ Extraídas {len(muestras_recortadas)} ráfagas temporales.")
# 2. INYECCIÓN A LA RED CAPA A CAPA
drones_encontrados = 0
ruidos_encontrados = 0
print("\n==============================================")
print("  REPORTE DETALLADO POR RÁFAGA FÍSICA         ")
print("==============================================")
with torch.no_grad():
    for i, pulso in enumerate(muestras_recortadas):
        
        # Inyectar AGC al vuelo si el conmutador está activado
        if USAR_MODELO_NORMALIZADO:
            pulso_final = pulso / torch.max(torch.abs(pulso))
        else:
            pulso_final = pulso / torch.max(torch.abs(pulso))
        
        input_ia = pulso_final.unsqueeze(0).to(device) 
        
        outputs = model(input_ia)
        prob = F.softmax(outputs, dim=1)
        _, prediccion = outputs.max(1)
        
        prob_dron = prob[0][1].item() * 100 
        
        # Recuperar marca temporal del Stage 1 físico
        t_ms_inicio = bursts[i]['t0']
        
        if prediccion.item() == 1:
            drones_encontrados += 1
            print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | 🧠 IA: DRON   [{prob_dron:6.2f}%] ✅")
        else:
            ruidos_encontrados += 1
            print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | 🧠 IA: RUIDO  [{prob_dron:6.2f}%] ❌")
print("\n==============================================")
if USAR_MODELO_NORMALIZADO:
    print("Modo: ESCALA MIN-MAX DINÁMICA (AGC ON)")
else:
    print("Modo: ESCALA CRUDA (AGC OFF - RIESGO DE OVERFITTING)")
print("----------------------------------------------")
print(f"Total Evaluadas : {len(muestras_recortadas)}")
print(f"Detecciones Dron: {drones_encontrados}")
print(f"Falsos Ruidos   : {ruidos_encontrados}")
print(f"Tasa de Acierto : {(drones_encontrados/len(muestras_recortadas))*100:.1f} %")
print("==============================================")
